## PlantGuard‑Meta — Kaggle GPU PPO Training (Online, uses HF Space reward)

This notebook is designed to **use Kaggle GPU effectively**:
- Loads a small instruct model in **4-bit** (fits on Kaggle GPU) and trains with **TRL PPO**.
- Interacts with the **deployed Hugging Face Space** over HTTP (`/reset`, `/step`).
- Uses the **environment’s reward** (not an offline heuristic).
- Logs episode metrics to `training_log.csv` and renders `reward_curve.png` + `success_rate.png`.

**HF Space runtime URL used here**: `https://narendra78-plant-governor-env.hf.space`

> If your Space URL differs (custom subdomain), change `BASE_URL` in the config cell.


In [ ]:
# --- Install (Kaggle) ---
# Kaggle usually has internet; if disabled, enable it in notebook settings.

%pip -q install -U \
  "transformers>=4.45" \
  "trl>=0.25.0" \
  "accelerate>=0.34" \
  "datasets>=2.20" \
  "bitsandbytes" \
  "torch" \
  "pandas" "numpy" "matplotlib" "requests"

import torch, sys, platform
print('python', sys.version.split()[0])
print('platform', platform.platform())
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))

# Verify installs early
import transformers
import trl
import accelerate
print('transformers', transformers.__version__)
print('trl', trl.__version__)
print('accelerate', accelerate.__version__)


In [ ]:
# --- Config ---
import os, json, time, csv, random
from dataclasses import dataclass, asdict
from typing import Any, Dict, Optional

import numpy as np
import pandas as pd
import requests

BASE_URL = "https://narendra78-plant-governor-env.hf.space"
RESET_URL = f"{BASE_URL}/reset"
STEP_URL = f"{BASE_URL}/step"
STATE_URL = f"{BASE_URL}/state"
HEALTH_URL = f"{BASE_URL}/health"

print('BASE_URL', BASE_URL)
print('health', requests.get(HEALTH_URL, timeout=30).json())

# Training knobs (start small; scale up on Kaggle GPU)
TOTAL_EPISODES = 50          # set 200+ for stronger curves
MAX_STEPS_PER_EPISODE = 200  # set 720 for full shift
SEED0 = 1234

# Model: Qwen (16GB GPU default)
# 7B in 4-bit + LoRA/GRPO is a strong sweet spot on 16GB.
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# If you hit OOM, fall back to:
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

OUT_DIR = "./outputs"
os.makedirs(OUT_DIR, exist_ok=True)
TRAIN_LOG_CSV = os.path.join(OUT_DIR, "training_log.csv")


In [ ]:
# --- HTTP helpers (no websockets; Kaggle-friendly) ---

def env_reset(seed: Optional[int] = None) -> Dict[str, Any]:
    payload: Dict[str, Any] = {}
    if seed is not None:
        payload["seed"] = int(seed)
    r = requests.post(RESET_URL, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()


def env_step(action: Dict[str, Any]) -> Dict[str, Any]:
    r = requests.post(STEP_URL, json=action, timeout=60)
    r.raise_for_status()
    return r.json()


def env_state() -> Dict[str, Any]:
    r = requests.get(STATE_URL, timeout=60)
    r.raise_for_status()
    return r.json()


def obs_to_prompt(obs: Dict[str, Any], step: int, last_action: str) -> str:
    return f"""You are PlantGuard‑Meta, an AI plant operations manager.

Step {step}/720 | Sensors:
- air_temp: {obs.get('air_temp', 0):.1f} K
- process_temp: {obs.get('process_temp', 0):.1f} K
- rotational_speed: {obs.get('rotational_speed', 0):.0f} rpm
- torque: {obs.get('torque', 0):.1f} Nm
- tool_wear: {obs.get('tool_wear', 0):.0f} min
- shift_hour: {obs.get('shift_hour', 0)}
- remaining_budget: ${obs.get('remaining_budget', 0):.0f}
- last_action: {last_action}

Choose ONE action tool and explain why.
Return ONLY valid JSON like:
{{"tool":"do_nothing","reasoning":"...","load_reduction":null}}

Valid tools:
- run_diagnostic
- adjust_load (optionally set load_reduction 0.1-0.9)
- dispatch_repair
- order_spare_part
- do_nothing
"""


VALID_TOOLS = {
    "run_diagnostic",
    "adjust_load",
    "dispatch_repair",
    "order_spare_part",
    "do_nothing",
}


def parse_action(text: str) -> Dict[str, Any]:
    t = (text or "").strip()
    if "```json" in t:
        t = t.split("```json", 1)[1].split("```", 1)[0].strip()
    elif "```" in t:
        t = t.split("```", 1)[1].split("```", 1)[0].strip()
    start = t.find("{")
    end = t.rfind("}") + 1
    if start != -1 and end > start:
        t = t[start:end]

    try:
        data = json.loads(t)
    except Exception:
        return {"tool": "do_nothing", "reasoning": "parse_error", "load_reduction": None}

    tool = data.get("tool", "do_nothing")
    reasoning = data.get("reasoning", "") or ""
    lr = data.get("load_reduction", None)

    if tool not in VALID_TOOLS:
        tool = "do_nothing"

    if tool == "adjust_load" and lr is not None:
        try:
            lr = float(lr)
            lr = max(0.1, min(0.9, lr))
        except Exception:
            lr = None
    else:
        lr = None

    return {"tool": tool, "reasoning": reasoning, "load_reduction": lr}


In [ ]:
# --- Load model (4-bit) + PPO trainer ---
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    device_map="auto",
)

ppo_config = PPOConfig(
    batch_size=1,
    mini_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    ppo_epochs=1,
    log_with=None,
)

trainer = PPOTrainer(
    config=ppo_config,
    model=model,
    tokenizer=tokenizer,
)

print('model loaded')


In [ ]:
# --- Online PPO loop (env reward) ---
@dataclass
class EpisodeMetrics:
    episode: int
    seed: int
    steps: int
    total_reward: float
    avg_reward: float
    cascade: bool
    complete: bool
    budget_remaining: float
    wall_time_s: float


def generate_response(prompt: str, max_new_tokens: int = 220) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.pretrained_model.device)
    with torch.no_grad():
        out = trainer.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
        )
    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen


random.seed(SEED0)
np.random.seed(SEED0)

log_rows = []
with open(TRAIN_LOG_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(asdict(EpisodeMetrics(0,0,0,0,0,False,False,0,0)).keys()))
    writer.writeheader()

for ep in range(TOTAL_EPISODES):
    t0 = time.time()
    seed = SEED0 + ep

    reset_payload = env_reset(seed=seed)
    obs = reset_payload.get("observation", reset_payload)  # tolerate either shape

    last_action = "none"
    total_reward = 0.0
    steps = 0

    while steps < MAX_STEPS_PER_EPISODE:
        prompt = obs_to_prompt(obs, steps, last_action)

        # PPO expects tensors for query/response
        query_t = tokenizer(prompt, return_tensors="pt").input_ids[0].to(model.pretrained_model.device)

        response_text = generate_response(prompt)
        action = parse_action(response_text)

        # Step the real environment
        step_payload = env_step(action)
        reward = float(step_payload.get("reward") or 0.0)
        done = bool(step_payload.get("done", False))

        obs = step_payload.get("observation", {})
        last_action = action.get("tool", "do_nothing")

        total_reward += reward
        steps += 1

        # PPO update step (single transition)
        response_t = tokenizer(response_text, return_tensors="pt").input_ids[0].to(model.pretrained_model.device)
        trainer.step([query_t], [response_t], [reward])

        if done:
            break

    st = env_state()
    wall = time.time() - t0

    m = EpisodeMetrics(
        episode=ep,
        seed=seed,
        steps=int(st.get("step_count", steps)),
        total_reward=float(total_reward),
        avg_reward=float(total_reward / max(steps, 1)),
        cascade=bool(st.get("cascade_occurred", False)),
        complete=bool(st.get("shift_complete", False)),
        budget_remaining=float(st.get("budget_remaining", obs.get("remaining_budget", 0.0))),
        wall_time_s=float(wall),
    )
    log_rows.append(m)

    with open(TRAIN_LOG_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(m).keys()))
        writer.writerow(asdict(m))

    print(f"ep={ep} steps={m.steps} reward={m.total_reward:+.1f} complete={m.complete} cascade={m.cascade} wall={m.wall_time_s:.1f}s")

print("\nWrote", TRAIN_LOG_CSV)


In [ ]:
# --- Plots (Kaggle working dir) ---
import matplotlib.pyplot as plt


df = pd.read_csv(TRAIN_LOG_CSV).sort_values('episode')

plt.figure(figsize=(8,4))
plt.plot(df['episode'], df['total_reward'], alpha=0.35, label='episode')
plt.plot(df['episode'], df['total_reward'].rolling(10, min_periods=2).mean(), label='moving avg (w=10)')
plt.xlabel('episode')
plt.ylabel('total reward')
plt.title('Reward curve (online PPO, env reward)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'reward_curve.png'), dpi=180)
plt.show()

plt.figure(figsize=(8,4))
plt.plot(df['episode'], df['complete'].astype(int).rolling(10, min_periods=2).mean())
plt.xlabel('episode')
plt.ylabel('rolling success rate')
plt.title('Shift completion rate (rolling)')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'success_rate.png'), dpi=180)
plt.show()

print('Saved plots in', OUT_DIR)


## Option B (Recommended): TRL GRPO training using *environment rewards*

This section follows the standard TRL pattern: **rollout → step environment → attach `env_reward` → train with GRPO**.

Notes:
- GRPO uses multiple generations per prompt; keep `NUM_GENERATIONS` small if your Space has limited concurrency.
- This is a minimal 1-step-per-prompt version (still uses the real environment reward). You can expand it to multi-step episodes later.


In [ ]:
# --- GRPO setup (Qwen + env rewards) ---
# If you ran PPO above, you can restart the kernel before running GRPO to free VRAM.

%pip -q install -U "trl>=0.25.0" "transformers>=4.45" "accelerate>=0.34" "datasets>=2.20" "bitsandbytes" "torch" "requests" "pandas" "numpy"

import json, random, time
from typing import Any, Dict, List

import numpy as np
import requests
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

# Qwen model (GPU-friendly). Change to 0.5B if you hit OOM.
MODEL_ID = MODEL_NAME  # uses the config cell's Qwen model

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    load_in_4bit=True,
    device_map="auto",
)

print('GRPO model loaded:', MODEL_ID)


In [ ]:
# --- Build a prompt dataset from real environment resets ---
# We create N initial observations (different seeds), convert them to prompts.

N_PROMPTS = 200  # for quick Kaggle runs; increase for better training

random.seed(SEED0)
np.random.seed(SEED0)

prompts: List[Dict[str, Any]] = []
for i in range(N_PROMPTS):
    seed = SEED0 + i
    reset_payload = env_reset(seed=seed)
    obs = reset_payload.get("observation", reset_payload)
    prompts.append({"prompt": obs_to_prompt(obs, step=0, last_action="none")})

train_ds = Dataset.from_list(prompts)
print('dataset size', len(train_ds))
print('\nSample prompt:\n', train_ds[0]['prompt'][:500])


In [ ]:
# --- Rollout function: generate → parse action → env.step → attach env_reward ---
from typing import Tuple


def rollout_func(prompts: List[str], args: GRPOConfig, processing_class):
    # `processing_class` is the tokenizer
    tok = processing_class

    completions = []
    env_rewards = []

    # For each prompt, we generate ONE completion here.
    # GRPO will call rollout multiple times across generations.
    for p in prompts:
        inputs = tok(p, return_tensors="pt", truncation=True).to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=220,
                do_sample=True,
                temperature=1.0,
                top_p=0.95,
                pad_token_id=tok.eos_token_id,
            )
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        action = parse_action(text)
        step_payload = env_step(action)
        r = float(step_payload.get("reward") or 0.0)

        # GRPO expects chat-like completion format in many examples.
        completions.append([{"content": text}])
        env_rewards.append(r)

    return {
        "completions": completions,
        "env_reward": env_rewards,
    }


def reward_from_env(prompts, completions, env_reward=None, **kwargs):
    # TRL passes `env_reward` from rollout_func into reward funcs via kwargs.
    # Return list[float]
    if env_reward is None:
        return [0.0 for _ in completions]
    return [float(x) for x in env_reward]


In [ ]:
# --- Train with GRPO ---
# Keep num_generations small if your Space cannot handle many concurrent sessions.

grpo_args = GRPOConfig(
    output_dir=os.path.join(OUT_DIR, "grpo_out"),
    max_steps=200,                  # increase (e.g., 1000+) for real improvement
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=2,
    max_completion_length=220,
    report_to="none",
    logging_steps=5,
    save_steps=50,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_from_env,
    train_dataset=train_ds,
    args=grpo_args,
    rollout_func=lambda prompts, args, processing_class: rollout_func(prompts, args, processing_class),
)

trainer.train()
print('GRPO done. Outputs in', grpo_args.output_dir)
